# Milestone 3 - Strong Scaling (MPI GMM)

Uses Milestone 2 distributed GMM as blueprint with additional Milestone 3 instrumentation.

Strong scaling: fixed dataset size, increasing MPI ranks.
Measured metrics: runtime, speedup, efficiency, iteration time, communication overhead, convergence, memory, I/O, and cluster quality/coherence.

## AWS EC2 Setup

Recommended instance families: c7i (compute), r7i (memory-heavy runs).

Install dependencies (Ubuntu):
```bash
sudo apt-get update
sudo apt-get install -y python3-pip python3-venv openmpi-bin libopenmpi-dev build-essential
python3 -m venv .venv
source .venv/bin/activate
pip install --upgrade pip
pip install numpy pandas scikit-learn mpi4py matplotlib psutil notebook
```

For fair MPI runs: set OMP/BLAS threads to 1.

In [ ]:
import os
import shlex
import subprocess
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

for _var in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"]:
    os.environ.setdefault(_var, "1")

def find_project_root(start=None):
    p = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [p] + list(p.parents):
        if (candidate / "Final Datasets").exists() and (candidate / "Milestone 3").exists():
            return candidate
    raise FileNotFoundError("Project root not found")

PROJECT_ROOT = find_project_root()
M3_DIR = PROJECT_ROOT / "Milestone 3"
RUNNER = M3_DIR / "m3_experiment_runner.py"
RESULTS_DIR = M3_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.dpi"] = 120
pd.set_option("display.max_columns", None)

print(PROJECT_ROOT)
print(RUNNER)

In [ ]:
EMBEDDING = "bge"
DISTRIBUTION = "block"  # block | cyclic | load_balanced
N_CLUSTERS = 10
FIXED_ROWS = 8000
RANKS = [1, 2, 4, 8]
N_INIT = 1
MAX_ITER = 120
TOL = 1e-3

OUT_CSV = RESULTS_DIR / "m3_strong_scaling_metrics.csv"
if OUT_CSV.exists():
    OUT_CSV.unlink()

print(f"Output: {OUT_CSV}")

In [ ]:
def run_mpi_experiment(ranks):
    cmd = [
        "mpirun", "-np", str(ranks),
        "python", str(RUNNER),
        "--mode", "strong",
        "--embedding", EMBEDDING,
        "--distribution", DISTRIBUTION,
        "--n-clusters", str(N_CLUSTERS),
        "--rows", str(FIXED_ROWS),
        "--n-init", str(N_INIT),
        "--max-iter", str(MAX_ITER),
        "--tol", str(TOL),
        "--tag", f"strong_p{ranks}",
        "--out-file", str(OUT_CSV),
    ]
    print("Running:", " ".join(shlex.quote(x) for x in cmd))
    subprocess.run(cmd, cwd=str(PROJECT_ROOT), check=True)

for p in RANKS:
    run_mpi_experiment(p)

print("Strong scaling runs complete.")

In [ ]:
df = pd.read_csv(OUT_CSV).sort_values("mpi_ranks").reset_index(drop=True)
baseline_time = float(df.loc[df["mpi_ranks"].idxmin(), "gmm_total_seconds"])
baseline_ranks = int(df["mpi_ranks"].min())

df["strong_speedup_x"] = baseline_time / df["gmm_total_seconds"]
df["strong_efficiency"] = df["strong_speedup_x"] / (df["mpi_ranks"] / baseline_ranks)
df["comm_overhead_pct"] = 100.0 * df["communication_fraction"]
df["convergence_rate_iter_per_s"] = df["gmm_iterations"] / df["gmm_total_seconds"]

display_cols = [
    "mpi_ranks", "rows_used", "n_clusters",
    "gmm_total_seconds", "avg_iteration_seconds",
    "communication_seconds", "comm_overhead_pct",
    "gmm_iterations", "convergence_rate_iter_per_s",
    "strong_speedup_x", "strong_efficiency",
    "silhouette_known", "purity", "ari", "jaccard_weighted",
    "coherence_top_share_mean", "coherence_entropy_mean",
    "peak_rss_mb_max_rank", "io_load_seconds", "io_throughput_mb_s"
]
print(df[display_cols].to_string(index=False))

df.to_csv(OUT_CSV, index=False)
print(f"Updated metrics saved to {OUT_CSV}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].plot(df["mpi_ranks"], df["gmm_total_seconds"], marker="o")
axes[0, 0].set_title("Runtime vs MPI Ranks")
axes[0, 0].set_xlabel("MPI ranks")
axes[0, 0].set_ylabel("seconds")
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(df["mpi_ranks"], df["strong_speedup_x"], marker="o", label="Measured")
axes[0, 1].plot(df["mpi_ranks"], df["mpi_ranks"] / df["mpi_ranks"].min(), linestyle="--", label="Ideal")
axes[0, 1].set_title("Strong Scaling Speedup")
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(df["mpi_ranks"], 100.0 * df["strong_efficiency"], marker="o")
axes[1, 0].set_title("Strong Scaling Efficiency (%)")
axes[1, 0].set_xlabel("MPI ranks")
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(df["mpi_ranks"], df["comm_overhead_pct"], marker="o", label="Comm overhead %")
axes[1, 1].plot(df["mpi_ranks"], df["avg_iteration_seconds"], marker="s", label="Avg iter sec")
axes[1, 1].set_title("Communication and Iteration")
axes[1, 1].set_xlabel("MPI ranks")
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plot_path = RESULTS_DIR / "m3_strong_scaling_plots.png"
plt.savefig(plot_path, bbox_inches="tight")
plt.show()
print(plot_path)

# Milestone 3 - Strong Scaling (MPI GMM)

This notebook runs **strong scaling** using the Milestone 2 distributed GMM blueprint and Milestone 3 instrumentation.

Strong scaling setup:
- Fixed dataset size
- Increasing MPI ranks
- Measure: total runtime, iteration time, communication overhead, convergence behavior, memory, I/O, and cluster quality/coherence metrics.

It uses already prepared embeddings from `Final Datasets` (no re-embedding needed).

## EC2 Requirements and Optimization

Recommended EC2:
- Start with `c7i.2xlarge` or `c7i.4xlarge` (compute-optimized, strong single-thread perf).
- If memory pressure appears, use `r7i.xlarge` or larger.

System packages:
- Python 3.10+
- OpenMPI (`mpirun`)
- Build tools for mpi4py

Python packages:
- numpy, pandas, scikit-learn, mpi4py, matplotlib, psutil, jupyter

Threading controls (important for MPI):
- `OMP_NUM_THREADS=1`
- `OPENBLAS_NUM_THREADS=1`
- `MKL_NUM_THREADS=1`
- `NUMEXPR_NUM_THREADS=1`

Example setup commands on Ubuntu EC2:
```bash
sudo apt-get update
sudo apt-get install -y python3-pip python3-venv openmpi-bin libopenmpi-dev build-essential
python3 -m venv .venv
source .venv/bin/activate
pip install --upgrade pip
pip install numpy pandas scikit-learn mpi4py matplotlib psutil notebook
```

Run notebook kernel with the same environment where mpi4py/OpenMPI is installed.

In [ ]:
import os
import shlex
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

for _var in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"]:
    os.environ.setdefault(_var, "1")

plt.rcParams["figure.dpi"] = 120
pd.set_option("display.max_columns", None)

def find_project_root(start=None):
    p = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [p] + list(p.parents):
        if (candidate / "Final Datasets").exists() and (candidate / "Milestone 3").exists():
            return candidate
    raise FileNotFoundError("Project root not found")

PROJECT_ROOT = find_project_root()
M3_DIR = PROJECT_ROOT / "Milestone 3"
RUNNER = M3_DIR / "m3_experiment_runner.py"
RESULTS_DIR = M3_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(PROJECT_ROOT)
print(RUNNER)

In [ ]:
# Experiment configuration
EMBEDDING = "bge"            # bge | tfidf | tfidf_lsa50
DISTRIBUTION = "block"       # block | cyclic | load_balanced
N_CLUSTERS = 10
FIXED_ROWS = 8000
RANKS = [1, 2, 4, 8]
N_INIT = 1
MAX_ITER = 120
TOL = 1e-3

OUT_CSV = RESULTS_DIR / "m3_strong_scaling_metrics.csv"
if OUT_CSV.exists():
    OUT_CSV.unlink()

print(f"Output: {OUT_CSV}")

In [ ]:
def run_mpi_experiment(ranks):
    cmd = [
        "mpirun", "-np", str(ranks),
        "python", str(RUNNER),
        "--mode", "strong",
        "--embedding", EMBEDDING,
        "--distribution", DISTRIBUTION,
        "--n-clusters", str(N_CLUSTERS),
        "--rows", str(FIXED_ROWS),
        "--n-init", str(N_INIT),
        "--max-iter", str(MAX_ITER),
        "--tol", str(TOL),
        "--tag", f"strong_p{ranks}",
        "--out-file", str(OUT_CSV),
    ]
    print("Running:", " ".join(shlex.quote(x) for x in cmd))
    subprocess.run(cmd, cwd=str(PROJECT_ROOT), check=True)

for p in RANKS:
    run_mpi_experiment(p)

print("Strong scaling runs complete.")

In [ ]:
df = pd.read_csv(OUT_CSV).sort_values("mpi_ranks").reset_index(drop=True)
baseline_time = float(df.loc[df["mpi_ranks"].idxmin(), "gmm_total_seconds"])
baseline_ranks = int(df["mpi_ranks"].min())

df["strong_speedup_x"] = baseline_time / df["gmm_total_seconds"]
df["strong_efficiency"] = df["strong_speedup_x"] / (df["mpi_ranks"] / baseline_ranks)
df["comm_overhead_pct"] = 100.0 * df["communication_fraction"]
df["convergence_rate_iter_per_s"] = df["gmm_iterations"] / df["gmm_total_seconds"]

display_cols = [
    "mpi_ranks", "rows_used", "n_clusters",
    "gmm_total_seconds", "avg_iteration_seconds",
    "communication_seconds", "comm_overhead_pct",
    "gmm_iterations", "convergence_rate_iter_per_s",
    "strong_speedup_x", "strong_efficiency",
    "silhouette_known", "purity", "ari", "jaccard_weighted",
    "coherence_top_share_mean", "coherence_entropy_mean",
    "peak_rss_mb_max_rank", "io_load_seconds", "io_throughput_mb_s"
]
display(df[display_cols])

df.to_csv(OUT_CSV, index=False)
print(f"Updated metrics saved to {OUT_CSV}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].plot(df["mpi_ranks"], df["gmm_total_seconds"], marker="o")
axes[0, 0].set_title("Runtime vs MPI Ranks")
axes[0, 0].set_xlabel("MPI ranks")
axes[0, 0].set_ylabel("seconds")
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(df["mpi_ranks"], df["strong_speedup_x"], marker="o", label="Measured")
axes[0, 1].plot(df["mpi_ranks"], df["mpi_ranks"] / df["mpi_ranks"].min(), linestyle="--", label="Ideal")
axes[0, 1].set_title("Strong Scaling Speedup")
axes[0, 1].set_xlabel("MPI ranks")
axes[0, 1].set_ylabel("speedup x")
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(df["mpi_ranks"], 100 * df["strong_efficiency"], marker="o")
axes[1, 0].set_title("Strong Scaling Efficiency")
axes[1, 0].set_xlabel("MPI ranks")
axes[1, 0].set_ylabel("efficiency %")
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(df["mpi_ranks"], df["comm_overhead_pct"], marker="o", label="Comm overhead %")
axes[1, 1].plot(df["mpi_ranks"], df["avg_iteration_seconds"], marker="s", label="Avg iter sec")
axes[1, 1].set_title("Communication and Iteration Metrics")
axes[1, 1].set_xlabel("MPI ranks")
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plot_path = RESULTS_DIR / "m3_strong_scaling_plots.png"
plt.savefig(plot_path, bbox_inches="tight")
plt.show()
print(plot_path)

## Notes

- For fair strong-scaling analysis, keep `FIXED_ROWS`, embedding, and `n_clusters` constant across all runs.
- To compare data distribution strategies, rerun this notebook with `DISTRIBUTION` set to `block`, `cyclic`, and `load_balanced` and compare `comm_overhead_pct` and runtime.